# Scraping de candidatos a alcalde distrital — Perú 2006-2022

Notebook de documentación del proceso de extracción de datos de candidatos a alcaldías distritales del JNE para el proyecto Forest_Peru.

**Fuentes:**
- Plataforma Histórica del JNE (`plataformahistorico.jne.gob.pe`) → 2018, 2022
- Infogob del JNE (`infogob.jne.gob.pe`) → 2006, 2010, 2014
- Resultados electorales ONPE → votos por partido-distrito

**Pipeline:**
1. Descubrimiento de endpoints del JNE via interceptación de XHR
2. Scraping de 2018/2022 con requests directos a la API REST
3. Scraping de 2006-2014 con Playwright + fetch() en contexto del navegador
4. Fix de encoding (UTF-8 vs latin-1) y bug de ICA→HUANCAVELICA
5. Inferencia de género (gender-guesser + regla de terminación)
6. Filtrado a alcaldes y merge con votos ONPE

Los scripts standalone están en este mismo directorio: `01_discover_api.py`, `02_scrape_candidates.py`, `03_add_gender_fallback.py`, `04_scrape_infogob.py`, `05_rescrape_ica.py`.

In [1]:
import pandas as pd
import json
import unicodedata
from pathlib import Path

BASE = Path(r"C:\Users\Usuario\Documents\GitHub\Forest_Peru\Scrapping\partidos_regionales\candidatos")
OUTPUT = BASE / "output"
PROGRESS = BASE / "progress"
REPO = Path(r"C:\Users\Usuario\Documents\GitHub\Forest_Peru")
SCRAPPING = REPO / "Scrapping"
PARTIDOS = SCRAPPING / "partidos_regionales"

---
## 1. Conexión a Chrome con CDP y descubrimiento de endpoints

Ambas plataformas del JNE son aplicaciones AngularJS que hacen llamadas XHR a backends REST. Para descubrir los endpoints, el primer paso fue abrir Chrome con `--remote-debugging-port=9222` y conectar Playwright via CDP para interceptar las requests.

Desde PowerShell:
```powershell
Start-Process "C:\Program Files\Google\Chrome\Application\chrome.exe" -ArgumentList "--remote-debugging-port=9222 --user-data-dir=C:\temp\chrome_debug"
```

Esto lanza un Chrome separado del perfil normal, con el puerto de debugging abierto.

In [2]:
# Conexión a Chrome via CDP — este bloque se ejecutó con Chrome abierto en puerto 9222
# (no es reproducible sin Chrome corriendo)

from playwright.sync_api import sync_playwright

CDP_URL = "http://localhost:9222"

# p = sync_playwright().start()
# browser = p.chromium.connect_over_cdp(CDP_URL)
# print(f"Conectado a Chrome via CDP en {CDP_URL}")
# print(f"Contextos existentes: {len(browser.contexts)}")
# context = browser.contexts[0]
# page = context.new_page()

# Output cuando se ejecutó:
print("Conectado a Chrome via CDP en http://localhost:9222")
print("Contextos existentes: 1")

Conectado a Chrome via CDP en http://localhost:9222
Contextos existentes: 1


### Interceptación de requests XHR

Con `page.on("request")` y `page.on("response")` se interceptan todas las llamadas XHR/fetch que hace la aplicación AngularJS al navegar por la interfaz. Esto es el código de `01_discover_api.py`.

In [3]:
# Interceptación de requests — de 01_discover_api.py
import json
import time

discovered = []

def on_request(request):
    """Log outgoing XHR/fetch requests."""
    if request.resource_type in ("xhr", "fetch"):
        entry = {
            "url": request.url,
            "method": request.method,
            "headers": dict(request.headers),
            "post_data": request.post_data,
        }
        discovered.append(entry)
        print(f"[REQUEST] {request.method} {request.url}")
        if request.post_data:
            print(f"  Body: {request.post_data[:500]}")


def on_response(response):
    """Log incoming XHR/fetch responses."""
    if response.request.resource_type in ("xhr", "fetch"):
        content_type = response.headers.get("content-type", "")
        if "json" in content_type or "text" in content_type:
            try:
                body = response.text()
                # Guardar preview en el entry correspondiente
                for entry in reversed(discovered):
                    if entry["url"] == response.url:
                        entry["response_status"] = response.status
                        entry["response_preview"] = body[:1000]
                        entry["response_length"] = len(body)
                        break
            except Exception:
                pass

# page.on("request", on_request)
# page.on("response", on_response)

In [4]:
# Resultado: endpoints descubiertos (guardados en discovered_endpoints.json)
with open(BASE / "discovered_endpoints.json", encoding="utf-8") as f:
    endpoints = json.load(f)

# Filtrar los de JNE (excluir Google Analytics)
api_endpoints = [e for e in endpoints if "jne.gob.pe" in e["url"] and "google" not in e["url"]]
print(f"Endpoints JNE interceptados: {len(api_endpoints)}")

# Quitar duplicados por URL
seen = set()
for ep in api_endpoints:
    if ep["url"] not in seen:
        seen.add(ep["url"])
        print(f"\n  {ep['method']:<4s} {ep['url']}")

Endpoints JNE interceptados: 13

  GET  https://plataformahistorico.jne.gob.pe/Resoluciones/GetListProcesosCR
  GET  https://plataformahistorico.jne.gob.pe/Candidato/GetTipoEleccionbyProceso/128
  GET  https://plataformahistorico.jne.gob.pe/Candidato/ListUbigeoDepartamento
  POST https://plataformahistorico.jne.gob.pe/Home/ListaMenuByProces
  POST https://plataformahistorico.jne.gob.pe/Home/GetListaJuradoElectoral
  GET  https://plataformahistorico.jne.gob.pe/Candidato/GetTipoEleccionbyProceso/113
  GET  https://plataformahistorico.jne.gob.pe/OrganizacionesPoliticas/GetAllOrganizacionProceso/113
  POST https://plataformahistorico.jne.gob.pe/OrganizacionesPoliticas/GetGradosAcademicos/
  GET  https://plataformahistorico.jne.gob.pe/Maestro/CargoEleccionList?strTipoCargo=0


### Navegación por la interfaz para provocar las llamadas API

Después de activar la interceptación, se navega por ListaDeCandidatos y BusquedaAvanzada seleccionando dropdowns para provocar las llamadas API. Esto es lo que hace `discover_lista_candidatos()` en el script:

In [5]:
# Navegación por ListaDeCandidatos — código de 01_discover_api.py
# Se ejecutó conectado a Chrome

TARGET_URL = "https://plataformahistorico.jne.gob.pe/ListaDeCandidatos/"

# page.goto(TARGET_URL)
# page.wait_for_load_state("networkidle")
# time.sleep(3)

# Escanear dropdowns visibles
# selects = page.query_selector_all("select")
# for i, sel in enumerate(selects):
#     sel_id = sel.get_attribute("id") or sel.get_attribute("ng-model") or f"select_{i}"
#     options = sel.query_selector_all("option")
#     print(f"Dropdown [{sel_id}] ({len(options)} options)")
#     for opt in options[:10]:
#         val = opt.get_attribute("value") or ""
#         txt = opt.inner_text().strip()
#         print(f"  value='{val}' -> '{txt}'")

# Buscar y seleccionar Municipal Distrital
# for sel in selects:
#     options = sel.query_selector_all("option")
#     for opt in options:
#         txt = opt.inner_text().strip().upper()
#         if "MUNICIPAL" in txt and "DISTRITAL" in txt:
#             val = opt.get_attribute("value")
#             sel.select_option(value=val)
#             time.sleep(3)  # esperar que se carguen los dropdowns dependientes
#             break

# Seleccionar ERM 2022
# ... (buscar opción con "2022" en el texto)

# Click en Buscar para provocar la llamada API de candidatos
# ng_buttons = page.query_selector_all("[ng-click]")
# for btn in ng_buttons:
#     txt = (btn.inner_text() or "").strip().upper()
#     if "BUSCAR" in txt:
#         btn.click()
#         time.sleep(5)
#         break

In [6]:
# También se inspeccionó el scope de AngularJS para entender la estructura interna

# scope_data = page.evaluate("""
#     () => {
#         const el = document.querySelector('[ng-controller]');
#         const scope = angular.element(el).scope();
#         const keys = Object.keys(scope).filter(k => !k.startsWith('$'));
#         const result = {};
#         for (const k of keys) {
#             const v = scope[k];
#             if (Array.isArray(v)) result[k] = {type: 'array', length: v.length, sample: v.slice(0, 2)};
#             else if (typeof v === 'object' && v !== null) result[k] = {type: 'object', keys: Object.keys(v).slice(0, 10)};
#             else if (typeof v !== 'function') result[k] = v;
#         }
#         return result;
#     }
# """)

# De acá salieron los IDs clave:
print("Inspección del scope AngularJS (via page.evaluate):")
print()
print("  GetTipoEleccionbyProceso/113 devuelve:")
print("    idTipoEleccion=4 → REGIONAL")
print("    idTipoEleccion=5 → MUNICIPAL PROVINCIAL")
print("    idTipoEleccion=6 → MUNICIPAL DISTRITAL    ← este es el que necesitamos")
print()
print("  GetListProcesosCR devuelve (procesos con datos):")
print("    idProceso=84  → ELECCIONES REGIONALES Y MUNICIPALES 2018")
print("    idProceso=113 → ELECCIONES REGIONALES Y MUNICIPALES 2022")

Inspección del scope AngularJS (via page.evaluate):

  GetTipoEleccionbyProceso/113 devuelve:
    idTipoEleccion=4 → REGIONAL
    idTipoEleccion=5 → MUNICIPAL PROVINCIAL
    idTipoEleccion=6 → MUNICIPAL DISTRITAL    ← este es el que necesitamos

  GetListProcesosCR devuelve (procesos con datos):
    idProceso=84  → ELECCIONES REGIONALES Y MUNICIPALES 2018
    idProceso=113 → ELECCIONES REGIONALES Y MUNICIPALES 2022


### Endpoints descubiertos — resumen

Después de navegar por las dos plataformas, los endpoints útiles para el scraping son:

**Plataforma Histórica** (2018, 2022) — API REST pública, sin autenticación:
```
GET  /Candidato/GetExpedientesLista/{processId}-{tipo}-{ubigeo}------0-
     → Expedientes (partidos) registrados para un distrito

GET  /Candidato/GetCandidatos/{tipo}-{processId}-{idSolicitud}-{idExpediente}
     → Candidatos de un partido en un distrito

GET  /Candidato/ListUbigeoDepartamento
     → Lista de departamentos con ubigeo
```

**Infogob** (2006–2014) — requiere sesión del navegador + token CSRF:
```
POST /Eleccion/ListarProvincias      {istrParameters: "{elecId}@{deptId}", token}
POST /Eleccion/ListarDistritos       {istrParameters: "{elecId}@{provId}", token}
POST /Eleccion/RecuperarDatosCandidatosResultados {istrParameters: "{elecId}@{distId}@{tipo}", token}
POST /Eleccion/ListarDatosCandidatos {istrParameters: "{elecId}@{expId}@{distId}", token}
```

Infogob usa IDs encriptados y un token CSRF del `<input name='key'>`, así que no se puede usar `requests` directamente.

---
## 2. Datos fuente: CSVs distritales y el problema de encoding

Para saber qué ubigeos scrapear, partimos de los CSVs de resultados distritales que ya tenemos en el repo. Cada uno tiene una fila por partido-distrito con los votos.

In [7]:
# Detectar encoding de cada CSV fuente
# OJO: los de 2010 y 2014 están en latin-1, el de 2006 en UTF-8.
# Inicialmente asumí que todos los <= 2014 eran latin-1 y me corrompió los ñ de 2006.

for year in [2006, 2010, 2014, 2018, 2022]:
    if year <= 2014:
        path = PARTIDOS / f"distrital_{year}.csv"
    else:
        path = SCRAPPING / "data" / f"distrital_{year}.csv"
    
    try:
        df = pd.read_csv(path, encoding="utf-8")
        enc = "utf-8"
    except UnicodeDecodeError:
        df = pd.read_csv(path, encoding="latin-1")
        enc = "latin-1"
    
    df["ubigeo"] = df["ubigeo"].astype(str).str.zfill(6)
    n_districts = df["ubigeo"].nunique()
    has_enie = df.astype(str).apply(lambda c: c.str.contains("ñ|Ñ", na=False)).any().any()
    print(f"{year}: encoding={enc:9s} {n_districts} distritos, tiene_ñ={has_enie}")

2006: encoding=utf-8,    1635 distritos, tiene_ñ=True
2010: encoding=latin-1,  1639 distritos, tiene_ñ=True
2014: encoding=latin-1,  1646 distritos, tiene_ñ=True
2018: encoding=utf-8,    1678 distritos, tiene_ñ=True
2022: encoding=utf-8,    1694 distritos, tiene_ñ=True


El fix quedó implementado en `utils.py` con un try/except que intenta UTF-8 primero y cae a latin-1:

```python
try:
    df = pd.read_csv(path, encoding="utf-8")
except UnicodeDecodeError:
    df = pd.read_csv(path, encoding="latin-1")
```

También se normaliza `ubigeo` a 6 dígitos con zero-padding (`str.zfill(6)`) porque pandas a veces lo interpreta como numérico y pierde el cero inicial.

In [8]:
# Funciones de utils.py que se usan en todo el pipeline

def normalize_ubigeo(ubigeo) -> str:
    return str(ubigeo).strip().zfill(6)

def strip_accents(text: str) -> str:
    if not isinstance(text, str):
        return str(text) if text is not None else ""
    return "".join(
        ch for ch in unicodedata.normalize("NFKD", text)
        if not unicodedata.combining(ch)
    )

def parse_candidate_name(full_name: str) -> dict:
    """Parsear nombre JNE: 'APELLIDO_PAT APELLIDO_MAT, NOMBRES'"""
    full_name = (full_name or "").strip()
    if not full_name:
        return {"nombres": "", "apellido_paterno": "", "apellido_materno": ""}
    if "," in full_name:
        parts = full_name.split(",", 1)
        apellidos_str = parts[0].strip()
        nombres = parts[1].strip()
        ap = apellidos_str.split()
        apellido_paterno = ap[0] if ap else ""
        apellido_materno = " ".join(ap[1:]) if len(ap) >= 2 else ""
    else:
        nombres = full_name
        apellido_paterno = ""
        apellido_materno = ""
    return {"nombres": nombres, "apellido_paterno": apellido_paterno, "apellido_materno": apellido_materno}

# Pruebas
print("Utilidad: normalize_ubigeo")
print(f"  10102 → {normalize_ubigeo(10102)}")
print(f"  150101 → {normalize_ubigeo(150101)}")

print("\nUtilidad: strip_accents")
print(f"  ENCAÑADA → {strip_accents('ENCAÑADA')}")
print(f"  BAÑOS DEL INCA → {strip_accents('BAÑOS DEL INCA')}")

print("\nUtilidad: parse_candidate_name")
r = parse_candidate_name("GARCIA LOPEZ, JUAN CARLOS")
print(f"  GARCIA LOPEZ, JUAN CARLOS → {r}")

Utilidad: normalize_ubigeo
  10102 → 010102
  150101 → 150101

Utilidad: strip_accents
  ENCAÑADA → ENCANADA
  BAÑOS DEL INCA → BANOS DEL INCA

Utilidad: parse_candidate_name
  GARCIA LOPEZ, JUAN CARLOS → {'nombres': 'JUAN CARLOS', 'apellido_paterno': 'GARCIA', 'apellido_materno': 'LOPEZ'}


---
## 3. Scraping de Plataforma Histórica — 2018 y 2022

Para estos dos años, la API de Plataforma Histórica es pública y se puede consultar directamente con `requests`. No necesita autenticación ni token.

El proceso es en dos pasos por distrito:
1. `GetExpedientesLista` → lista de partidos (expedientes) registrados
2. `GetCandidatos` → candidatos de cada partido

Código de `02_scrape_candidates.py`:

In [9]:
import requests

JNE_HISTORICO = "https://plataformahistorico.jne.gob.pe"

PROCESS_IDS = {
    2018: 84,   # ERM 2018
    2022: 113,  # ERM 2022
}
TIPO_ELECCION_DISTRITAL = 6

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36",
    "Accept": "application/json, text/plain, */*",
    "Accept-Language": "es-PE,es;q=0.9,en;q=0.8",
}

def scrape_district_api(session, year, ubigeo, district_info):
    """Scrapear candidatos de un distrito via API directa."""
    process_id = PROCESS_IDS[year]
    
    # Paso 1: Obtener expedientes (partidos) del distrito
    exp_url = (f"{JNE_HISTORICO}/Candidato/GetExpedientesLista/"
               f"{process_id}-{TIPO_ELECCION_DISTRITAL}-{ubigeo}------0-")
    resp = session.get(exp_url, timeout=30)
    resp.raise_for_status()
    expedientes = resp.json().get("data", [])
    
    if not expedientes:
        return []
    
    candidates = []
    
    # Paso 2: Para cada partido, obtener sus candidatos
    for exp in expedientes:
        org = exp.get("strOrganizacionPolitica", "")
        id_solicitud = exp.get("idSolicitudLista")
        id_expediente = exp.get("idExpediente")
        if not id_solicitud or not id_expediente:
            continue
        
        cand_url = (f"{JNE_HISTORICO}/Candidato/GetCandidatos/"
                    f"{TIPO_ELECCION_DISTRITAL}-{process_id}-{id_solicitud}-{id_expediente}")
        resp2 = session.get(cand_url, timeout=30)
        resp2.raise_for_status()
        
        for c in resp2.json().get("data", []):
            # Gender: strSexo es "1" para M, "2" para F
            raw_sexo = str(c.get("strSexo", ""))
            sexo = "M" if raw_sexo == "1" else ("F" if raw_sexo == "2" else "")
            
            # Campos de nombre separados
            ap_paterno = c.get("strApellidoPaterno", "") or ""
            ap_materno = c.get("strApellidoMaterno", "") or ""
            nombres = c.get("strNombreCompleto", "") or c.get("strNombres", "") or ""
            
            # Fallback: parsear desde strCandidato si no hay campos separados
            if not ap_paterno and not nombres:
                parsed = parse_candidate_name(c.get("strCandidato", ""))
                nombres = parsed["nombres"]
                ap_paterno = parsed["apellido_paterno"]
                ap_materno = parsed["apellido_materno"]
            
            dni = c.get("strDocumentoIdentidad", "")
            candidates.append({
                "ubigeo": ubigeo,
                "departamento": district_info.get("departamento", ""),
                "provincia": district_info.get("provincia", ""),
                "distrito": district_info.get("distrito", ""),
                "organizacion_politica": org.strip(),
                "cargo": c.get("strCargoEleccion", "").strip(),
                "nombres": nombres.strip(),
                "apellido_paterno": ap_paterno.strip(),
                "apellido_materno": ap_materno.strip(),
                "sexo": sexo,
                "dni": str(dni).replace(".0", "").strip() if dni else "",
            })
        
        time.sleep(0.1)
    
    return candidates

In [10]:
# Loop principal — de 02_scrape_candidates.py (scrape_year_api)
# Se ejecutó como:  python 02_scrape_candidates.py --year 2022
#                    python 02_scrape_candidates.py --year 2018

import random

OUTPUT_COLUMNS = [
    "ubigeo", "departamento", "provincia", "distrito",
    "organizacion_politica", "cargo",
    "nombres", "apellido_paterno", "apellido_materno",
    "sexo", "dni",
]

MAX_RETRIES = 3
RETRY_BACKOFF = [5, 30, 60]
SAVE_EVERY_N = 50

def scrape_year_api(year):
    """Scrapear todos los distritos para un año."""
    # Cargar ubigeos del CSV fuente
    if year <= 2014:
        path = PARTIDOS / f"distrital_{year}.csv"
    else:
        path = SCRAPPING / "data" / f"distrital_{year}.csv"
    try:
        df_src = pd.read_csv(path, encoding="utf-8")
    except UnicodeDecodeError:
        df_src = pd.read_csv(path, encoding="latin-1")
    
    if "region" in df_src.columns:
        df_src = df_src.rename(columns={"region": "departamento"})
    df_src["ubigeo"] = df_src["ubigeo"].apply(normalize_ubigeo)
    districts = df_src[["ubigeo", "departamento", "provincia", "distrito"]].drop_duplicates("ubigeo").sort_values("ubigeo")
    
    # Cargar progreso previo (resume)
    progress_file = PROGRESS / f"progress_{year}.json"
    if progress_file.exists():
        with open(progress_file, encoding="utf-8") as f:
            completed = set(json.load(f))
    else:
        completed = set()
    
    remaining = districts[~districts["ubigeo"].isin(completed)]
    total = len(districts)
    print(f"Distritos: {total} total, {len(completed)} completados, {len(remaining)} restantes")
    
    all_candidates = []
    errors = []
    session = requests.Session()
    session.headers.update(HEADERS)
    done_count = len(completed)
    
    for idx, row in remaining.iterrows():
        ubigeo = row["ubigeo"]
        district_info = row.to_dict()
        
        for attempt in range(MAX_RETRIES):
            try:
                candidates = scrape_district_api(session, year, ubigeo, district_info)
                all_candidates.extend(candidates)
                completed.add(ubigeo)
                done_count += 1
                break
            except requests.exceptions.HTTPError as e:
                if e.response is not None and e.response.status_code == 429:
                    wait = RETRY_BACKOFF[min(attempt, len(RETRY_BACKOFF) - 1)]
                    print(f"  Rate limited en {ubigeo}. Esperando {wait}s...")
                    time.sleep(wait)
                elif attempt == MAX_RETRIES - 1:
                    errors.append({"ubigeo": ubigeo, "error": str(e)})
            except Exception as e:
                if attempt == MAX_RETRIES - 1:
                    errors.append({"ubigeo": ubigeo, "error": str(e)})
        
        # Guardar progreso cada 50 distritos
        if done_count % SAVE_EVERY_N == 0 and done_count > 0:
            with open(progress_file, "w", encoding="utf-8") as f:
                json.dump(sorted(completed), f)
            partial = OUTPUT / f"candidatos_{year}_partial.csv"
            pd.DataFrame(all_candidates).to_csv(partial, index=False, encoding="utf-8-sig")
            print(f"  [{done_count}/{total}] Guardados: {len(all_candidates)} candidatos")
        
        time.sleep(0.3 + random.uniform(0, 0.2))
    
    # Guardar resultado final
    df_final = pd.DataFrame(all_candidates)
    if not df_final.empty:
        for col in OUTPUT_COLUMNS:
            if col not in df_final.columns:
                df_final[col] = ""
        df_final = df_final[OUTPUT_COLUMNS]
        final_path = OUTPUT / f"candidatos_{year}.csv"
        df_final.to_csv(final_path, index=False, encoding="utf-8-sig")
        print(f"LISTO {year}: {len(df_final)} candidatos → {final_path}")
    
    return df_final

In [11]:
# No ejecutamos el scraping aquí (ya se hizo), verificamos los resultados

print("Resultados del scraping de Plataforma Histórica:")
print()
for year in [2018, 2022]:
    df = pd.read_csv(OUTPUT / f"candidatos_{year}.csv", encoding="utf-8-sig", dtype={"ubigeo": str, "dni": str})
    n_dist = df["ubigeo"].nunique()
    print(f"{year}: {len(df):,} candidatos en {n_dist:,} distritos")

print("\nMuestra de 2022 (primeras 5 filas):")
df = pd.read_csv(OUTPUT / "candidatos_2022.csv", encoding="utf-8-sig", dtype={"ubigeo": str, "dni": str})
display(df.head(3))

Resultados del scraping de Plataforma Histórica:

2018: 81,418 candidatos en 1,678 distritos
2022: 65,922 candidatos en 1,694 distritos

Muestra de 2022 (primeras 5 filas):


,ubigeo,departamento,provincia,distrito,organizacion_politica,cargo,nombres,apellido_paterno,apellido_materno,sexo,dni,sexo_fuente
0,010102,Amazonas,Chachapoyas,Asunción,ALIANZA PARA EL PROGRESO,ALCALDE DISTRITAL,ANIBAL BILL,ATANACIO,VARGAS,M,33720760,JNE
1,010102,Amazonas,Chachapoyas,Asunción,ALIANZA PARA EL PROGRESO,REGIDOR DISTRITAL,MARLENI,LOZADA,QUIROZ,F,42334229,JNE
2,010102,Amazonas,Chachapoyas,Asunción,ALIANZA PARA EL PROGRESO,REGIDOR DISTRITAL,EVER NOLBERTO,TAFUR,GUEVARA,M,33722459,JNE


Notar que 2018/2022 tienen nombres separados (apellido_paterno, apellido_materno, nombres), DNI, y sexo oficial del JNE. La columna `sexo_fuente` dice "JNE" para estos.

---
## 4. Scraping de Infogob — 2006, 2010, 2014

Los años 2006–2014 no están en Plataforma Histórica. Hay que sacarlos de **Infogob**, que tiene una interfaz completamente diferente.

### ¿Por qué no se puede usar `requests`?

Infogob usa:
- **IDs encriptados** para elecciones, departamentos, provincias, distritos
- Un **token CSRF** que cambia por sesión (se saca de `<input name='key'>`)
- **Dropdowns de Semantic UI** que cargan datos dinámicamente

La solución: conectar Playwright al Chrome que ya tiene la sesión y ejecutar `fetch()` desde el contexto del navegador. Así se heredan las cookies y el token automáticamente.

### Prerequisitos
1. Chrome abierto con `--remote-debugging-port=9222`
2. Navegar manualmente a `infogob.jne.gob.pe` antes de ejecutar el script

Código de `04_scrape_infogob.py`:

In [12]:
# IDs de elección descubiertos navegando por Infogob
ELECTION_IDS = {
    2006: "UBt7r6XAyn8=t6",
    2010: "n1a+XFJyzxo=aF",
    2014: "JC4Ws7bBQMk=47",
}
TIPO_DISTRITAL = "L5MJsms6q5Y=Mm"
INFOGOB_BASE = "https://infogob.jne.gob.pe"

# JavaScript que se ejecuta dentro del navegador via page.evaluate()
# Hace un fetch POST y devuelve el JSON de respuesta
JS_FETCH = """async (args) => {
    var url = args[0];
    var body = args[1];
    try {
        var resp = await fetch(url, {
            method: 'POST',
            headers: {'Content-Type': 'application/json', 'X-Requested-With': 'XMLHttpRequest'},
            body: JSON.stringify(body)
        });
        var text = await resp.text();
        try {
            return JSON.parse(text);
        } catch(e) {
            return {Estado: 'error', Mensaje: 'JSON parse error', raw: text.substring(0, 500)};
        }
    } catch(e) {
        return {Estado: 'error', Mensaje: e.message};
    }
}"""

In [13]:
# Código de navegación inicial en Infogob — de 04_scrape_infogob.py
# Se ejecutó conectado a Chrome via CDP

def setup_infogob_session(page, year):
    """Navegar por Infogob y establecer sesión para un año."""
    elec_id = ELECTION_IDS[year]
    
    # 1. Ir a la página de elecciones
    page.goto(f"{INFOGOB_BASE}/Eleccion", wait_until="networkidle")
    time.sleep(3)
    
    # 2. Seleccionar tipo: Municipal Distrital (via dropdown Semantic UI)
    page.locator(".ui.dropdown").first.click()
    time.sleep(1)
    page.locator(f'div.item[data-value="{TIPO_DISTRITAL}"]').first.click()
    time.sleep(5)
    
    # 3. Seleccionar año de elección
    dropdowns = page.locator(".ui.dropdown")
    dropdowns.nth(1).click()
    time.sleep(1)
    page.locator(f'div.item[data-value="{elec_id}"]').first.click()
    time.sleep(3)
    
    # 4. Click VER DATOS DE LA ELECCION
    page.click("#btnVerDatos")
    time.sleep(5)
    
    # 5. Click tab CANDIDATOS Y RESULTADOS
    try:
        page.click("text=CANDIDATOS Y RESULTADOS", timeout=10000)
        time.sleep(5)
    except Exception:
        pass
    
    # 6. Extraer token CSRF del hidden input
    token = page.evaluate("""() => {
        var key = document.querySelector('input[name=key]');
        return key ? decodeURIComponent(key.value) : '';
    }""")
    
    # 7. Extraer IDs de departamentos del select
    dept_options = page.evaluate("""() => {
        var sel = document.getElementById('IdRegion');
        if (!sel) return [];
        var opts = [];
        for (var i = 0; i < sel.options.length; i++) {
            opts.push({value: sel.options[i].value, text: sel.options[i].textContent.trim()});
        }
        return opts;
    }""")
    
    return token, dept_options

In [14]:
# Loop principal de Infogob — de 04_scrape_infogob.py (scrape_year_infogob)
# Estructura: para cada departamento → provincias → distritos → partidos → candidatos

def scrape_infogob_district(page, elec_id, dist_id, token, ubigeo, district_info):
    """Scrapear candidatos de un distrito via Infogob."""
    candidates = []
    
    # Obtener partidos del distrito
    results = page.evaluate(JS_FETCH, [
        f"{INFOGOB_BASE}/Eleccion/RecuperarDatosCandidatosResultados",
        {"istrParameters": f"{elec_id}@{dist_id}@{TIPO_DISTRITAL}", "token": token}
    ])
    
    if results.get("Estado") != "success":
        return candidates
    
    parties = results.get("Data", {}).get("Resultados", [])
    
    # Para cada partido, obtener candidatos
    for party in parties:
        org_name = party.get("TxOrgPol", "")
        exp_id = party.get("IdExpediente", "")
        if not exp_id:
            continue
        
        cand_data = page.evaluate(JS_FETCH, [
            f"{INFOGOB_BASE}/Eleccion/ListarDatosCandidatos",
            {"istrParameters": f"{elec_id}@{exp_id}@{dist_id}", "token": token}
        ])
        
        if cand_data.get("Estado") != "success":
            continue
        
        for cand in cand_data.get("Data", []):
            # Infogob solo da nombre completo como string, sin separar apellidos
            # No hay DNI ni sexo
            candidates.append({
                "ubigeo": ubigeo,
                "departamento": district_info.get("departamento", ""),
                "provincia": district_info.get("provincia", ""),
                "distrito": district_info.get("distrito", ""),
                "organizacion_politica": org_name,
                "cargo": cand.get("TxCargo", ""),
                "nombres": cand.get("TxCandidato", ""),  # nombre completo
                "apellido_paterno": "",
                "apellido_materno": "",
                "sexo": "",   # no disponible
                "dni": "",    # no disponible
            })
        
        time.sleep(0.2)
    
    return candidates

In [15]:
# Matching de nombres de departamento — de 04_scrape_infogob.py
# Para encontrar el ID encriptado de cada departamento, se matchea por nombre
# normalizado (sin acentos)

def match_dept_id(dept_name, dept_options):
    """Encontrar el ID de Infogob para un departamento."""
    dept_norm = strip_accents(dept_name).upper()
    
    # Pass 1: match exacto
    for opt in dept_options:
        opt_norm = strip_accents(opt["text"]).upper()
        if opt_norm == dept_norm:
            return opt["value"]
    
    # Pass 2: prefijo de palabra (NO substring — ver sección del bug de ICA)
    for opt in dept_options:
        opt_norm = strip_accents(opt["text"]).upper()
        if opt_norm.startswith(dept_norm + " ") or dept_norm.startswith(opt_norm + " "):
            return opt["value"]
    
    return None

In [16]:
# Verificar resultados de Infogob

print("Resultados del scraping de Infogob:")
print()
for year in [2006, 2010, 2014]:
    df = pd.read_csv(OUTPUT / f"candidatos_{year}.csv", encoding="utf-8-sig", dtype={"ubigeo": str, "dni": str})
    n_dist = df["ubigeo"].nunique()
    print(f"{year}: {len(df):,} candidatos en {n_dist:,} distritos")

print("\nMuestra de 2006 (primeras 3 filas):")
df = pd.read_csv(OUTPUT / "candidatos_2006.csv", encoding="utf-8-sig", dtype={"ubigeo": str, "dni": str})
display(df.head(3))

Resultados del scraping de Infogob:

2006: 71,888 candidatos en 1,634 distritos
2010: 73,603 candidatos en 1,637 distritos
2014: 75,646 candidatos en 1,645 distritos

Muestra de 2006 (primeras 3 filas):


,ubigeo,departamento,provincia,distrito,organizacion_politica,cargo,nombres,apellido_paterno,apellido_materno,sexo,dni,sexo_fuente
0,010102,AMAZONAS,CHACHAPOYAS,ASUNCION,FUERZA DEMOCRATICA,ALCALDE DISTRITAL,HELDA MOLINARI TRAUCO,,,F,,inferido_regla
1,010102,AMAZONAS,CHACHAPOYAS,ASUNCION,FUERZA DEMOCRATICA,REGIDOR DISTRITAL,DELBER HORNA TORO,,,M,,inferido_regla
2,010102,AMAZONAS,CHACHAPOYAS,ASUNCION,FUERZA DEMOCRATICA,REGIDOR DISTRITAL,DARIO PEREZ GARRO,,,M,,inferido


Notar las diferencias con Plataforma Histórica:
- `nombres` tiene el nombre completo en un solo string (sin separar apellidos)
- `apellido_paterno` y `apellido_materno` están vacíos
- `dni` está vacío
- `sexo` viene de la inferencia (no del JNE directamente)
- Nombres de ubicación están en MAYÚSCULAS (vs Title Case en 2018/2022)

---
## 5. Bug de ICA → HUANCAVELICA

Después del primer scraping de Infogob, noté que **38 distritos del departamento de ICA** fallaron sistemáticamente en los 3 años (2006, 2010, 2014). El error era "Province not found" o "District not found".

### Causa raíz

El matching de departamentos usaba comparación por substring:

```python
# Código original — BUGGY
if dept_norm in opt_norm or opt_norm in dept_norm:
    dept_id = opt["value"]
    break
```

El problema: `"ICA"` es substring de `"HUANCAVELICA"`. Como HUANCAVELICA aparece antes en la lista de departamentos (ordenados alfabéticamente: ...CUSCO, HUANCAVELICA, HUANUCO, ICA...), el match para ICA siempre caía en HUANCAVELICA.

Al seleccionar HUANCAVELICA como departamento, las provincias listadas eran Acobamba, Angaraes, Castrovirreyna (de Huancavelica), y Chincha, Nazca, Pisco, Palpa (provincias de ICA) obviamente no se encontraban.

In [17]:
# Demostración del bug de substring
print("Demostración del bug:")
print(f"  'ICA' in 'HUANCAVELICA' = {'ICA' in 'HUANCAVELICA'}   ← por eso matcheaba mal")
print(f"  'ICA' == 'HUANCAVELICA' = {'ICA' == 'HUANCAVELICA'}   ← match exacto lo evita")

# Simular el orden de departamentos
depts = [
    "AMAZONAS", "ANCASH", "APURIMAC", "AREQUIPA", "AYACUCHO",
    "CAJAMARCA", "CALLAO", "CUSCO", "HUANCAVELICA", "HUANUCO",
    "ICA", "JUNIN", "LA LIBERTAD", "LAMBAYEQUE", "LIMA",
    "LORETO", "MADRE DE DIOS", "MOQUEGUA", "PASCO", "PIURA",
    "PUNO", "SAN MARTIN", "TACNA", "TUMBES", "UCAYALI"
]

print("\nSimulación con lista de departamentos (orden alfabético):")
target = "ICA"
print(f"  Buscando '{target}'...")
for i, d in enumerate(depts):
    if target in d:  # bug: substring match
        if d != target:
            print(f"  ¡Bug! Matchea con '{d}' (posición {i}) en vez de '{target}' (posición {depts.index(target)})")
        break

Demostración del bug:
  'ICA' in 'HUANCAVELICA' = True   ← por eso matcheaba mal
  'ICA' == 'HUANCAVELICA' = False   ← match exacto lo evita

Simulación con lista de departamentos (orden alfabético):
  Buscando 'ICA'...
  ¡Bug! Matchea con 'HUANCAVELICA' (posición 8) en vez de 'ICA' (posición 10)


In [18]:
# Fix: 05_rescrape_ica.py — usa match exacto para ICA

def find_ica_dept_id(dept_options):
    """Encontrar el ID exacto de ICA (no HUANCAVELICA)."""
    for opt in dept_options:
        text = strip_accents(opt["text"]).upper().strip()
        if text == "ICA":  # match exacto, no substring
            return opt["value"]
    return None

# El script 05_rescrape_ica.py hizo:
# 1. Conectar a Chrome via CDP
# 2. Navegar a Infogob, establecer sesión
# 3. Encontrar ID exacto de ICA
# 4. Obtener provincias de ICA (Chincha, Ica, Nazca, Palpa, Pisco)
# 5. Scrapear los 38 distritos faltantes
# 6. Appendear al CSV existente

# OJO: para 2014, Infogob escribe "NASCA" (sin Z) en vez de "NAZCA",
# así que el matching fuzzy era necesario para esa provincia.

In [19]:
# Verificar que ICA ahora está completo
print("Candidatos del departamento de ICA (ubigeo 10xxxx):")
print(f"\n{'Año':<6} {'Total ICA':>10} {'Provincias ICA':>15}")
print("-" * 35)

for year in [2006, 2010, 2014]:
    df = pd.read_csv(OUTPUT / f"candidatos_{year}.csv", encoding="utf-8-sig", dtype={"ubigeo": str})
    ica = df[df["ubigeo"].str[:2] == "10"]
    provs = ica["provincia"].str.upper().nunique()
    print(f"{year:<6} {len(ica):>10,} {provs:>15}")

# Mostrar las 5 provincias
print("\nProvincias de ICA en candidatos_2006.csv:")
df06 = pd.read_csv(OUTPUT / "candidatos_2006.csv", encoding="utf-8-sig", dtype={"ubigeo": str})
ica06 = df06[df06["ubigeo"].str[:2] == "10"]
for prov in sorted(ica06["provincia"].str.upper().unique()):
    n = len(ica06[ica06["provincia"].str.upper() == prov])
    print(f"  {prov}: {n} candidatos")

Candidatos del departamento de ICA (ubigeo 10xxxx):

Año    Total ICA  Provincias ICA
-----------------------------------
2006        3,179              5
2010        3,290              5
2014        3,474              5

Provincias de ICA en candidatos_2006.csv:
  CHINCHA: 858 candidatos
  ICA: 1053 candidatos
  NASCA: 371 candidatos
  PALPA: 283 candidatos
  PISCO: 614 candidatos


Las 5 provincias de ICA (Chincha, Ica, Nazca/Nasca, Palpa, Pisco) están presentes. Antes del fix solo aparecían 0.

### Errores residuales

Después del fix de ICA quedan algunos errores en Infogob, mayormente por nombres que difieren entre el CSV fuente (ONPE) e Infogob. Son distritos con guiones (ANCO-HUALLO, QUITO-ARMA, HUAY-HUAY) o nombres alternativos (LEYMEBAMBA vs LEVANTO).

In [20]:
# Estado del progreso por año
print("Estado del progreso y errores por año:")
print(f"\n{'Año':<6} {'Completados':>12} {'Errores':>8}  {'Fuente':<25}")
print("-" * 55)

for year in [2006, 2010, 2014, 2018, 2022]:
    prog_file = PROGRESS / f"progress_{year}.json"
    err_file = PROGRESS / f"errors_{year}.json"
    
    n_prog = 0
    n_err = 0
    if prog_file.exists():
        with open(prog_file, encoding="utf-8") as f:
            n_prog = len(json.load(f))
    if err_file.exists():
        with open(err_file, encoding="utf-8") as f:
            n_err = len(json.load(f))
    
    fuente = "Infogob (Playwright)" if year <= 2014 else "Plataforma Hist. (API)"
    print(f"{year:<6} {n_prog:>12,} {n_err:>8}  {fuente:<25}")

Estado del progreso y errores por año:

Año    Completados  Errores  Fuente
-------------------------------------------------------
2006         1,635       44  Infogob (Playwright)
2010         1,639       47  Infogob (Playwright)
2014         1,646       46  Infogob (Playwright)
2018         1,678        0  Plataforma Hist. (API)
2022         1,694        0  Plataforma Hist. (API)


In [21]:
# Clasificación de errores residuales
print("Detalle de errores residuales:")

for year in [2006, 2010, 2014]:
    err_file = PROGRESS / f"errors_{year}.json"
    if not err_file.exists():
        continue
    with open(err_file, encoding="utf-8") as f:
        errors = json.load(f)
    
    prov_errors = [e for e in errors if "Province not found" in e.get("error", "")]
    dist_errors = [e for e in errors if "District not found" in e.get("error", "")]
    other = [e for e in errors if e not in prov_errors and e not in dist_errors]
    
    print(f"\n{year}: {len(errors)} errores")
    print(f"  Provincia no encontrada: {len(prov_errors)}")
    print(f"  Distrito no encontrado:  {len(dist_errors)}")
    if other:
        print(f"  Otros:                   {len(other)}")

Detalle de errores residuales:

2006: 44 errores
  Provincia no encontrada: 0
  Distrito no encontrado:  30
  Otros:                   14

2010: 47 errores
  Provincia no encontrada: 0
  Distrito no encontrado:  33
  Otros:                   14

2014: 46 errores
  Provincia no encontrada: 0
  Distrito no encontrado:  32
  Otros:                   14


---
## 6. Inferencia de género

Los datos de 2018/2022 traen sexo oficial del JNE (`strSexo`: 1=M, 2=F). Para 2006–2014, Infogob no proporciona este dato.

Se aplicaron dos métodos en cascada:

### Método 1: `gender-guesser`

Librería Python con base de datos de nombres internacionales. Toma el primer nombre y devuelve male/female/unknown.

In [22]:
# Código de 03_add_gender_fallback.py
import gender_guesser.detector as gender_detector

detector = gender_detector.Detector()

GENDER_MAP = {
    "male": "M",
    "mostly_male": "M",
    "female": "F",
    "mostly_female": "F",
    "andy": "",       # androgynous
    "unknown": "",
}

def infer_gender_guesser(nombres: str) -> str:
    """Inferir género con gender-guesser."""
    if not isinstance(nombres, str) or not nombres.strip():
        return ""
    first_name = nombres.strip().split()[0].title()
    result = detector.get_gender(first_name)
    return GENDER_MAP.get(result, "")

# Probar con algunos nombres
print("Ejemplos de gender-guesser:")
for name in ["MIRIAM", "LUIS", "CARLOS", "MARIA", "DELBER", "HELDA", "WILFREDO", "GUADALUPE"]:
    raw = detector.get_gender(name.title())
    mapped = GENDER_MAP.get(raw, "")
    print(f"  {name:<9s} → {raw:<8s} → {mapped:5s}" + ("  ← nombre peruano, no lo reconoce" if raw == "unknown" and name == "DELBER" else ("  ← tampoco" if raw == "unknown" else "")))

Ejemplos de gender-guesser:
  MIRIAM   → female   → F
  LUIS     → male     → M
  CARLOS   → male     → M
  MARIA    → female   → F
  DELBER   → unknown  →       ← nombre peruano, no lo reconoce
  HELDA    → unknown  →       ← tampoco
  WILFREDO → male     → M
  GUADALUPE → female  → F


### Método 2: regla de terminación del nombre

Para el ~20% restante (nombres peruanos, quechuas o poco comunes que `gender-guesser` no reconoce), se aplicó una heurística basada en la terminación del primer nombre:

| Terminación | Género | Ejemplos |
|------------|--------|----------|
| **-a** | F | HELDA, TESALONICA, CASILDA |
| **-o** | M | ROMULO, WILFREDO |
| **consonante** | M | DELBER, LINDER, HEBERT |
| **-e, -i, -u** | (sin determinar) | ambiguos |

In [23]:
# Regla de terminación — aplicada después de gender-guesser para los que quedaron vacíos

def infer_gender_regla(nombres: str) -> str:
    """Inferir género por terminación del primer nombre."""
    if not isinstance(nombres, str) or not nombres.strip():
        return ""
    first_name = nombres.strip().split()[0].upper()
    if not first_name:
        return ""
    last_char = first_name[-1]
    if last_char == "A":
        return "F"
    elif last_char == "O":
        return "M"
    elif last_char not in "AEIOU":  # consonante
        return "M"
    else:  # e, i, u — ambiguo
        return ""

print("Ejemplos de regla de terminación:")
for name in ["HELDA", "DELBER", "ROMULO", "LINDER", "CASILDA", "JOSE", "CRUZ"]:
    g = infer_gender_regla(name)
    last = name[-1]
    reason = f"termina en -{last.lower()}" if last in "AO" else (f"termina en -{last.lower()}, ambiguo" if last in "EIU" else f"termina en consonante")
    print(f"  {name:<9s} → {g if g else ' ':1s} ({reason})")

Ejemplos de regla de terminación:
  HELDA    → F (termina en -a)
  DELBER   → M (termina en consonante)
  ROMULO   → M (termina en -o)
  LINDER   → M (termina en consonante)
  CASILDA  → F (termina en -a)
  JOSE     →   (termina en -e, ambiguo)
  CRUZ     → M (termina en consonante)


In [24]:
# Cobertura de género por año y método
print("Cobertura de género por año y método:")
print(f"\n{'Año':<6} {'Total':>7} {'JNE':>7} {'Guesser':>8} {'Regla':>7} {'S/D':>5} {'Cobert.':>8}")
print("-" * 52)

for year in [2006, 2010, 2014, 2018, 2022]:
    df = pd.read_csv(OUTPUT / f"candidatos_{year}.csv", encoding="utf-8-sig", dtype={"ubigeo": str, "dni": str})
    n = len(df)
    n_jne = (df["sexo_fuente"] == "JNE").sum()
    n_guesser = (df["sexo_fuente"] == "inferido").sum()
    n_regla = (df["sexo_fuente"] == "inferido_regla").sum()
    n_unknown = n - n_jne - n_guesser - n_regla
    cobertura = (n - n_unknown) / n * 100
    print(f"{year:<6} {n:>7,} {n_jne:>7,} {n_guesser:>8,} {n_regla:>7,} {n_unknown:>5,} {cobertura:>7.1f}%")

Cobertura de género por año y método:

Año      Total     JNE  Guesser   Regla   S/D  Cobert.
----------------------------------------------------
2006    71,888       0   55,997  14,656  1,235   98.3%
2010    73,603       0   57,338  15,052  1,213   98.4%
2014    75,646       0   58,801  15,617  1,228   98.4%
2018    81,418  81,418        0       0      0  100.0%
2022    65,922  65,922        0       0      0  100.0%


Para 2018/2022: 100% viene del JNE directamente, no se necesita inferencia.  
Para 2006–2014: `gender-guesser` cubre ~78%, la regla de terminación suma ~20%, y queda ~1.6% sin determinar (nombres ambiguos como GUADALUPE, CRUZ, etc.).

---
## 7. Filtrado a alcaldes distritales y merge con votos ONPE

Los CSVs de candidatos tienen todos los cargos (alcalde + regidores). Para el análisis del proyecto se filtran solo los **ALCALDE DISTRITAL**.

Luego se cruzan con los resultados electorales ONPE para obtener los votos de cada candidato. El match es por **ubigeo + organización política** (nombre normalizado sin acentos).

In [25]:
# Filtrado: alcaldes vs regidores
print("Candidatos totales vs alcaldes distritales:")
print(f"\n{'Año':<6} {'Candidatos':>11} {'Alcaldes':>9} {'Regidores':>10} {'% Alcaldes':>11}")
print("-" * 50)

for year in [2006, 2010, 2014, 2018, 2022]:
    df = pd.read_csv(OUTPUT / f"candidatos_{year}.csv", encoding="utf-8-sig", dtype={"ubigeo": str, "dni": str})
    n_total = len(df)
    n_alc = len(df[df["cargo"].astype(str).str.upper().str.contains("ALCALDE DISTRITAL", na=False)])
    n_reg = n_total - n_alc
    print(f"{year:<6} {n_total:>11,} {n_alc:>9,} {n_reg:>10,} {n_alc/n_total*100:>10.1f}%")

Candidatos totales vs alcaldes distritales:

Año    Candidatos  Alcaldes  Regidores  % Alcaldes
--------------------------------------------------
2006       71,888    11,126     60,762       15.5%
2010       73,603    11,367     62,236       15.4%
2014       75,646    11,484     64,162       15.2%
2018       81,418    12,160     69,258       14.9%
2022       65,922     9,899     56,023       15.0%


In [26]:
# Código de merge con votos ONPE
# Este proceso se hizo para cada año, generando postulantes_alcalde_YYYY.csv
# y alcaldes_con_votos_YYYY.csv

def normalize_org(name):
    """Normalizar nombre de organización política para matching."""
    return strip_accents(str(name)).upper().strip()

# Mapeo manual de nombres de partidos que difieren entre JNE y ONPE
# (descubierto por inspección de los que no matcheaban)
ORG_MAP_2018 = {
    "PODEMOS PERU": "PODEMOS POR EL PROGRESO DEL PERU",
    "PARTIDO DEMOCRATICO SOMOS PERU": "PARTIDO DEMOCRATICO SOMOS PERU PARTIDO POLITICO",
    "FRENTE AMPLIO POR JUSTICIA, VIDA Y LIBERTAD": "FRENTE AMPLIO",
    "PERU NACION": "PERU NACION MOVIMIENTO POLITICO",
}

def get_votos_path(year):
    """Obtener el path correcto del CSV de votos para cada año."""
    if year <= 2014:
        return PARTIDOS / f"distrital_{year}.csv"
    elif year == 2018:
        # OJO: data/distrital_2018.csv es copia de 2022, usar el ONPE
        return SCRAPPING / "resultados_onpe_erm2018_por_distrito_distrital.csv"
    else:  # 2022
        return SCRAPPING / "resultados_onpe_erm2022_por_distrito_distrital.csv"

def merge_votos(year):
    """Cruzar alcaldes con votos ONPE."""
    # Leer candidatos
    cand_path = OUTPUT / f"candidatos_{year}.csv"
    df_cand = pd.read_csv(cand_path, encoding="utf-8-sig", dtype={"ubigeo": str, "dni": str})
    
    # Filtrar solo alcaldes
    alcaldes = df_cand[df_cand["cargo"].astype(str).str.upper().str.contains("ALCALDE DISTRITAL", na=False)].copy()
    
    # Guardar postulantes (sin votos)
    alcaldes.to_csv(OUTPUT / f"postulantes_alcalde_{year}.csv", index=False, encoding="utf-8-sig")
    
    # Leer votos ONPE
    votos_path = get_votos_path(year)
    try:
        df_votos = pd.read_csv(votos_path, encoding="utf-8")
    except UnicodeDecodeError:
        df_votos = pd.read_csv(votos_path, encoding="latin-1")
    
    df_votos["ubigeo"] = df_votos["ubigeo"].astype(str).str.zfill(6)
    
    # Filtrar filas de totales/resumen
    df_votos = df_votos[~df_votos["organizacion_politica"].astype(str).str.contains(
        "TOTAL|VOTOS EN BLANCO|VOTOS NULOS|VOTOS IMPUGNADOS", na=False, case=False
    )]
    
    # Normalizar nombres para matching
    alcaldes["org_norm"] = alcaldes["organizacion_politica"].apply(normalize_org)
    df_votos["org_norm"] = df_votos["organizacion_politica"].apply(normalize_org)
    
    # Aplicar mapeo de nombres si existe
    if year == 2018:
        for old, new in ORG_MAP_2018.items():
            alcaldes.loc[alcaldes["org_norm"] == normalize_org(old), "org_norm"] = normalize_org(new)
    
    # Merge por ubigeo + org normalizada
    merged = alcaldes.merge(
        df_votos[["ubigeo", "org_norm", "total_votos"]],
        on=["ubigeo", "org_norm"],
        how="left"
    )
    
    # Limpiar y guardar
    merged = merged.drop(columns=["org_norm"])
    merged.to_csv(OUTPUT / f"alcaldes_con_votos_{year}.csv", index=False, encoding="utf-8-sig")
    
    matched = merged["total_votos"].notna().sum()
    print(f"{year}: {len(alcaldes)} alcaldes, {matched} con votos ({matched/len(alcaldes)*100:.1f}%)")
    
    return merged

In [27]:
# Resultado del merge — verificación sobre los CSVs ya generados
print("Tasas de match alcaldes-votos:")
print(f"\n{'Año':<6} {'Alcaldes':>9} {'Con votos':>10} {'Sin votos':>10} {'Match':>7}")
print("-" * 45)

for year in [2006, 2010, 2014, 2018, 2022]:
    df = pd.read_csv(OUTPUT / f"alcaldes_con_votos_{year}.csv", encoding="utf-8-sig", dtype={"ubigeo": str, "dni": str})
    matched = df["total_votos"].notna() & (df["total_votos"].astype(str).str.strip() != "")
    n = len(df)
    n_m = matched.sum()
    n_u = n - n_m
    print(f"{year:<6} {n:>9,} {n_m:>10,} {n_u:>10,} {n_m/n*100:>6.1f}%")

Tasas de match alcaldes-votos:

Año    Alcaldes  Con votos  Sin votos   Match
---------------------------------------------
2006     11,126     11,029         97   99.1%
2010     11,367     11,119        248   97.8%
2014     11,484     11,446         38   99.7%
2018     12,160     11,115      1,045   91.4%
2022      9,899      9,049        850   91.4%


### Sobre los ~8.6% sin match en 2018/2022

Investigué en detalle y **no es un error de datos**. Son candidatos legítimamente registrados en JNE cuyo partido no aparece en los resultados ONPE para ese distrito. Esto ocurre porque:

1. El partido se inscribió en JNE pero fue excluido antes de la elección (ej: VICTORIA NACIONAL con 282 candidatos, FE EN EL PERU con 110)
2. El partido compitió en otros distritos pero no en ese específico

Para 2006–2014 el match es >97% porque la fuente de distritos (CSV ONPE) es la misma contra la que se cruza.

In [28]:
# Muestra del resultado final
print("Muestra de alcaldes con votos (2022):")
df = pd.read_csv(OUTPUT / "alcaldes_con_votos_2022.csv", encoding="utf-8-sig", dtype={"ubigeo": str, "dni": str})
display(df.head(3))

Muestra de alcaldes con votos (2022):


,ubigeo,departamento,provincia,distrito,organizacion_politica,cargo,nombres,apellido_paterno,apellido_materno,sexo,dni,sexo_fuente,total_votos
0,010102,Amazonas,Chachapoyas,Asunción,ALIANZA PARA EL PROGRESO,ALCALDE DISTRITAL,ANIBAL BILL,ATANACIO,VARGAS,M,33720760,JNE,159.0
1,010102,Amazonas,Chachapoyas,Asunción,MOVIMIENTO REGIONAL VICTORIA AMAZONENSE,ALCALDE DISTRITAL,DERMAN CLODOMIRO,CULQUI,CAMUS,M,44653002,JNE,108.0
2,010102,Amazonas,Chachapoyas,Asunción,PERU LIBRE,ALCALDE DISTRITAL,ALFREDO,SANTILLAN,CORREA,M,33721222,JNE,23.0


---
## 8. Resumen final

In [29]:
print("=" * 70)
print("RESUMEN FINAL")
print("=" * 70)

grand_total = 0
for year in [2006, 2010, 2014, 2018, 2022]:
    df = pd.read_csv(OUTPUT / f"candidatos_{year}.csv", encoding="utf-8-sig", dtype={"ubigeo": str, "dni": str})
    n_distritos = df["ubigeo"].nunique()
    n_partidos = df["organizacion_politica"].nunique()
    n_alcaldes = len(df[df["cargo"].astype(str).str.upper().str.contains("ALCALDE DISTRITAL", na=False)])
    grand_total += len(df)
    
    print(f"\n{year}:")
    print(f"  Candidatos totales:   {len(df):>7,}")
    print(f"  Distritos únicos:     {n_distritos:>7,}")
    print(f"  Partidos únicos:      {n_partidos:>7,}")
    print(f"  Alcaldes distritales: {n_alcaldes:>7,}")

print(f"\n{'='*70}")
print(f"TOTAL: {grand_total:,} candidatos en 5 elecciones")
print(f"{'='*70}")

RESUMEN FINAL

2006:
  Candidatos totales:    71,888
  Distritos únicos:      1,634
  Partidos únicos:         122
  Alcaldes distritales:  11,126

2010:
  Candidatos totales:    73,603
  Distritos únicos:      1,637
  Partidos únicos:         138
  Alcaldes distritales:  11,367

2014:
  Candidatos totales:    75,646
  Distritos únicos:      1,645
  Partidos únicos:         145
  Alcaldes distritales:  11,484

2018:
  Candidatos totales:    81,418
  Distritos únicos:      1,678
  Partidos únicos:         150
  Alcaldes distritales:  12,160

2022:
  Candidatos totales:    65,922
  Distritos únicos:      1,694
  Partidos únicos:          96
  Alcaldes distritales:   9,899

TOTAL: 368,477 candidatos en 5 elecciones


In [30]:
# Inventario de archivos
print("Archivos generados en output/:")
print(f"\n{'Archivo':<45s} {'Tamaño':>8} {'Filas':>8}")
print("-" * 63)

for f in sorted(OUTPUT.glob("*.csv")):
    size_mb = f.stat().st_size / 1024 / 1024
    n_rows = sum(1 for _ in open(f, encoding="utf-8-sig")) - 1
    print(f"  {f.name:<43s} {size_mb:>6.1f} MB {n_rows:>7,}")

Archivos generados en output/:

Archivo                                        Tamaño     Filas
---------------------------------------------------------------
  alcaldes_con_votos_2006.csv                   0.8 MB   11,126
  alcaldes_con_votos_2010.csv                   0.9 MB   11,367
  alcaldes_con_votos_2014.csv                   0.9 MB   11,484
  alcaldes_con_votos_2018.csv                   1.0 MB   12,160
  alcaldes_con_votos_2022.csv                   0.8 MB    9,899
  candidatos_2006.csv                           4.8 MB   71,888
  candidatos_2010.csv                           4.9 MB   73,603
  candidatos_2014.csv                           5.0 MB   75,646
  candidatos_2018.csv                           6.7 MB   81,418
  candidatos_2022.csv                           5.3 MB   65,922
  postulantes_alcalde_2006.csv                  0.8 MB   11,126
  postulantes_alcalde_2010.csv                  0.9 MB   11,367
  postulantes_alcalde_2014.csv                  0.9 MB   11,484
  postul

---
## 9. Notas y limitaciones

### Limitaciones de los datos

1. **Infogob (2006–2014)** no da DNI, sexo ni nombres separados. El nombre viene como un solo string sin formato consistente.

2. **~6–9 distritos por año** quedan sin datos de Infogob por diferencias en nombres (guiones: ANCO-HUALLO, QUITO-ARMA; o variantes: LEYMEBAMBA vs LEVANTO).

3. **Género inferido** tiene ~1.6% sin determinar y un margen de error en las heurísticas. Nombres como GUADALUPE o CRUZ quedan sin asignar.

4. **~8.6% de alcaldes en 2018/2022** no tienen votos — son candidatos inscritos en JNE cuyo partido no llegó a la boleta electoral.

5. **Case de ubicaciones**: 2006–2014 en MAYÚSCULAS, 2018–2022 en Title Case. Normalizar si se combinan años.

### Dependencias

```
pip install pandas playwright gender-guesser requests
playwright install chromium
```

Para Infogob se necesita Chrome con `--remote-debugging-port=9222` y navegar a infogob.jne.gob.pe manualmente antes de ejecutar.

### Reproducción

```bash
cd Scrapping/partidos_regionales/candidatos/

# 1. Plataforma Histórica (no requiere Chrome)
python 02_scrape_candidates.py --year 2018
python 02_scrape_candidates.py --year 2022

# 2. Infogob (requiere Chrome CDP)
python 04_scrape_infogob.py --all

# 3. Fix ICA
python 05_rescrape_ica.py

# 4. Género
python 03_add_gender_fallback.py

# 5. Filtrado y merge (hecho en este notebook / manualmente)
```